In [29]:
from clyngor import ASP
from IPython.display import display, Markdown

In [30]:
with open('file1.lp', 'r') as f:
    program1 = f.read()
with open('file2.lp', 'r') as f:
    program2 = f.read()

In [31]:
# Function to convert answer sets to valid ASP facts
def answer_set_to_facts(answer_set,n,first):
    formatted_facts = []
    transmit= [("believe",2),("believe",3),("beliefbase",1), ("principle",1)]
    augment = [("utter",4),("act",1),("totalUti",2),("permissible",1), ("impermissible",1),('objective_lie',2),('objective_truth',2),('erroneous_lie',2),('erroneous_truth',2),('trigUti',3)]
    renamedAug = {("violated",1):"locally_violated"}
    for fact in answer_set:
        # Handle complex terms
        name = fact[0]
        args = fact[1] if isinstance(fact[1], tuple) else (fact[1],)
        arity = len(args)
        if first and (name,arity) in transmit :
          formatted_facts.append(f"{name}({','.join(map(str, args))}).")
        if (name,arity) in augment:
          newargs = ('s'+str(n),)+args
          formatted_facts.append(f"{name}({','.join(map(str, newargs))}).")
        if (name,arity) in renamedAug:
          newargs = ('s'+str(n),)+args
          newname = renamedAug[(name,arity)]
          formatted_facts.append(f"{newname}({','.join(map(str, newargs))}).")
    return "\n".join(formatted_facts)

# Run program1 to get its answer sets
answers_program1 = ASP(program1)
combined_program = ""

# Process each answer set from program1
for n, answer_set in enumerate(answers_program1, start=1):
    # print(f"Processing Answer Set {n} of program1...")

    # Convert answer set to ASP facts
    facts_from_program1 = answer_set_to_facts(answer_set,n,(n==1))

    # Combine program1 and program2
    combined_program = combined_program+"%AS"+str(n)+"\n"+f"{facts_from_program1}\n"

combined_program=combined_program+"\n"+program2

open('combined_program.lp', 'w').write(combined_program)

3905

In [32]:
answers_program2 = ASP(combined_program)

def get_all_facts(answer_set):
  # retrieves all the facts from the answer set
  facts = []
  for answer in answer_set:
    for fact in answer:
      facts.append(fact)
  return facts

def get_args(fact):
  # normalizes the arguments of a Clyngor fact in tuple
  return fact[1] if isinstance(fact[1], tuple) else (fact[1],)

def get_pred(facts, scenario):
  # retrieves the facts related to a scenario
  res = []
  for fact in facts:
    args = get_args(fact)
    if len(args) > 0 and args[0] == scenario:
      res.append(fact)
  return res

def get_acts(facts_s):
  # retrieves the actions
  res = []
  for fact in facts_s:
    if fact[0] in ("utter", "act"):
      args = get_args(fact)
      act_str = args[-1]
      if act_str != "evade(p,q)":
        res.append(act_str)
  return res

def has_evade(facts_s):
  #  indicates whether an escape attempt has taken place 
  for fact in facts_s:
    if fact[0] == "act":
      args = get_args(fact)
      if args[-1] == "evade(p,q)":
        return True
  return False

def get_perm(facts_s, act_str):
  res = dict()
  for fact in facts_s:
    args = get_args(fact)
    if fact[0] == 'permissible':
      res[args[-1]] = 'perm'
    elif fact[0] == 'impermissible':
      res[args[-1]] = 'imp'
    elif fact[0] in ["erroneous_truth", "erroneous_lie", "objective_truth", "objective_lie"] and args[-1] in act_str:
      res["rule"] = fact[0]
  res.setdefault("rule", "----")
  for formalism in ["deontologism", "principialism1", "principialism2", "consequentialism1", "consequentialism2"]:
    res.setdefault(formalism, "----")
  return res

def get_total_uti(facts_s, agent):
  # recovers the total utility for an agent (pBelief or env) in a scenario
  for fact in facts_s:
    if fact[0] == 'totalUti':
      args = get_args(fact)
      if args[1] == agent:
        return args[-1]
  return "----"

def get_events(facts_s, agent, include_evade=False):
  # retrieves events (kill/harm) triggered according to an agent’s beliefs
  events = []
  for fact in facts_s:
    if fact[0] == 'trigUti':
      args = get_args(fact)
      if args[1] == agent:
        events.append(str(args[2]))
  if include_evade and has_evade(facts_s):
    events.append("evade(p,q)")
  return events

facts = get_all_facts(answers_program2)

def build_big_table(all_facts, scenarios):
  header = ("| Scenario | Act | Rule | deontologism | principialism1 | principialism2 "
            "| consequentialism1 | consequentialism2 | attempt evade "
            "| uti (pBelief) | uti (env) "
            "| events (pBelief) | events (env) |\n")
  sep = "|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|\n"
  mk = header + sep

  for scenario in scenarios:
    facts_s = get_pred(all_facts, scenario)
    acts = get_acts(facts_s)
    evade = "True" if has_evade(facts_s) else "False"
    total_pbelief = get_total_uti(facts_s, "pBelief")
    total_env = get_total_uti(facts_s, "env")
    events_pbelief = get_events(facts_s, "pBelief", include_evade=True)
    events_env = get_events(facts_s, "env", include_evade=False)
    ev_pbelief_str = ", ".join(events_pbelief) if events_pbelief else "----"
    ev_env_str = ", ".join(events_env) if events_env else "----"

    if not acts:
      mk += (f"| {scenario} | ---- | ---- | ---- | ---- | ---- | ---- | ---- | {evade} "
             f"| {total_pbelief} | {total_env} | {ev_pbelief_str} | {ev_env_str} |\n")
      continue

    for act_str in acts:
      perm = get_perm(facts_s, act_str)
      mk += (f"| {scenario} | {act_str} | {perm['rule']} | {perm['deontologism']} "
             f"| {perm['principialism1']} | {perm['principialism2']} "
             f"| {perm['consequentialism1']} | {perm['consequentialism2']} "
             f"| {evade} | {total_pbelief} | {total_env} "
             f"| {ev_pbelief_str} | {ev_env_str} |\n")
  return mk

scenarios = [f"s{i}" for i in range(1, 6)]
mk_big = build_big_table(facts, scenarios)
display(Markdown(mk_big))


| Scenario | Act | Rule | deontologism | principialism1 | principialism2 | consequentialism1 | consequentialism2 | attempt evade | uti (pBelief) | uti (env) | events (pBelief) | events (env) |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| s1 | inform(at(r,madrid)) | objective_lie | imp | perm | perm | imp | perm | False | -3 | -3 | harm(q,p,uncoveredLie) | harm(q,p,uncredibleStatement) |
| s2 | inform(at(r,madrid)) | objective_lie | imp | perm | perm | perm | imp | True | 0 | -6 | evade(p,q) | harm(q,p,failedEvade), harm(q,p,uncredibleStatement) |
| s3 | inform(at(r,cemetery)) | erroneous_lie | imp | perm | perm | imp | imp | False | -3 | -4 | harm(q,p,uncoveredLie) | kill(q,p,r) |
| s4 | silence(p,q,0) | ---- | perm | ---- | ---- | imp | ---- | False | -3 | -3 | harm(q,p,refusedToAnswer) | harm(q,p,refusedToAnswer) |
| s5 | inform(at(r,family)) | erroneous_truth | imp | imp | imp | imp | perm | False | -4 | -3 | kill(q,p,r) | harm(q,p,uncoveredLie) |


The results obtained differ from those presented in Table 2 of the article concerning consequentialism 1. The implementation incorporates the scenario "say Madrid and try to escape," contrary to the article. As the permissions of consequentialist theories vary according to the scenarios considered, this leads to these divergences. Notably, in the scenario of "objective lie" without evasion, this act of language is authorized by consequentialism 1 in the article. However, since we are integrating here a scenario with an escape attempt, which has a higher utility according to Pablo’s belief base, this act becomes impermissible. Similarly, the consideration of scenario S2 leads to the impermissibility of scenario S3 for consequentialism 1.